# Código de Document Intelligence - OCR

## Librerías

In [1]:
# ===============================
# Librerías Estándar de Python
# ===============================
import copy
import io
import json
import os
import re
import shutil
import time
from time import sleep
from html import unescape
from itertools import permutations
import xml.etree.ElementTree as ET
from glob import glob
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# ===============================
# Librerías de Terceros
# ===============================
import fitz
import google.generativeai as genai
from bs4 import BeautifulSoup
from fuzzywuzzy import fuzz
import jsonschema
import openpyxl
import pandas as pd
import pdfplumber
from PyPDF2 import PdfReader
import requests
from PIL import Image

# ===============================
# Librerías de Entorno
# ===============================
from dotenv import load_dotenv

# ===============================
# Librerías para Pydantic
# ===============================
from enum import Enum
from pydantic import BaseModel, Field, constr, conlist, ValidationError
from typing import List, Optional, Literal, Annotated, Dict
import os
import io
import glob
import pandas as pd
from PIL import Image
import google.generativeai as genai

import os
import io
import glob
import pandas as pd
from PIL import Image
import google.generativeai as genai

## Paths y API Keys

In [4]:
path = os.path.join(os.environ['USERPROFILE'], "PATH")

## Cargar API KEY
load_dotenv(dotenv_path=os.path.join(path, "Códigos", ".env"))
api_key_google = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=api_key_google)


✅ Cliente de Google Gemini configurado correctamente.

CPU times: user 1.99 ms, sys: 1.55 ms, total: 3.54 ms
Wall time: 2.78 ms


# Path de interés

In [7]:
### Path general ###
path = os.path.join('SUB_PATH')

# Cambiar según corresponda
region = "Metropolitana"
cbr = "SANTIAGO"

### Path donde están los archivos PDF
data_path = os.path.join(path, region, cbr)

### Path donde dejar los resultados
result_path = os.path.join(data_path, "Resultados")


✅ Rutas de interés--OK.

CPU times: user 470 μs, sys: 153 μs, total: 623 μs
Wall time: 546 μs


## Función de Extracción

In [8]:
def build_prompt_for_page(page_num: int, total_pages: int) -> str:
    """
    Devuelve el prompt específico según si es la primera página o una página posterior.
    """
    if page_num == 0:
        # --- PROMPT PARA LA PÁGINA 1 (AJUSTADO) ---
        return f"""
    Eres un asistente experto en analizar documentos del Conservador de Bienes Raíces de Chile.
    Analiza la imagen de la PRIMERA PÁGINA de esta inscripción y extrae el texto, 
    clasificándolo rigurosamente con las siguientes etiquetas EXACTAS.
    Si una sección no existe, omite la etiqueta. NO inventes etiquetas.

    [Resumen inscripción:]
    (Texto principal de la carátula o resumen. Ej: 'COPIA AUTORIZADA', 'Conservador de Bienes Raíces de Puente Alto', 'Certifico que el presente documento...', 'Carátula Nro...')

    [Nota al margen izquierdo:]
    (Texto que aparezca claramente en el margen izquierdo. Ej: 'Nº 317', 'HERENCIA', 'Repertorio 11374' etc.)

    [Nota al margen derecho:]
    (Texto que aparezca claramente en el margen derecho. Ej: 'Según Certificado del Servicio de Impuestos Internos...')

    [Sello circular izquierdo:]
    (Texto contenido dentro del sello circular en la parte inferior izquierda. Ej: 'VERIFICAR DIRECTAMENTE EN WWW.FOJAS.CL...')

    [Sello circular derecho:]
    (Texto contenido dentro del sello circular en la parte inferior derecha. Ej: 'SERGIO YABER LOZANO CONSERVADOR...')

    [Código de barras:]
    (Si hay un código de barras, escribe [Código de barras])

    [Nota inferior:]
    (Texto en la parte inferior de la página, fuera de los sellos. Incluye 'Nro Certificado...', 'www.fojas.cl', y el párrafo 'Emito el presente documento...')

    [Firma electrónica:]
    (Texto relacionado con la firma digital si es visible. Ej: 'Firmado digitalmente por: SERGIO RODRIGO YABER LOZANO...')

    [manuscrita:]
    (Cualquier texto escrito a mano. Ej: [manuscrita: firma ilegible])


    INSTRUCCIONES IMPORTANTES:
    **- IMPORTANTE: Tu respuesta debe comenzar *directamente* con la primera etiqueta (ej: `[Nota al margen izquierdo:]`). NO incluyas ninguna introducción, saludo o frase previa como 'Claro, aquí tienes...'.**
    - Si un bloque NO aparece en la imagen, escribe justo debajo del bloque: (sin contenido detectado)
    - "Notas al margen" son los textos verticales o en la columna de la izquierda/derecha, como títulos, repertorios, FR, etc.
    - "Resumen inscripción" es el texto central de la primera página (el certificado que dice que la copia es fiel, fojas, número, etc.)
    - "Sello circular ..." escribe el texto completo que veas dentro del sello, incluso si está algo curvo.
    - "Código de barras" incluye las líneas que lo describen (p. ej. “Nro Certificado 3204295.- ...”)
    - "Nota inferior" es el párrafo que suele explicar la firma electrónica avanzada y la verificación en www.fojas.cl
    - "Firma electrónica" es el bloque de la firma digital (nombre del conservador, fecha, razón, ubicación). Si no viene separado, déjalo en Nota inferior.
    - Mantén tildes, puntos, guiones y mayúsculas tal como aparecen.
    - No agregues comentarios tuyos.
    - No consideres marcas de agua.
    - No elimines marcas como “Puente Alto, 05 de Diciembre de 2024.-”.
    - El documento está en Chile.
    """
    else:
        # --- PROMPT PARA PÁGINAS 2 Y SIGUIENTES (AJUSTADO) ---
        return f"""
    Eres un asistente experto en analizar documentos del Conservador de Bienes Raíces de Chile.
    Analiza la imagen de esta PÁGINA POSTERIOR (NO la primera) y extrae el texto, 
    clasificándolo rigurosamente con las siguientes etiquetas EXACTAS.
    Si una sección no existe, omite la etiqueta. NO inventes etiquetas.

    [Encabezado:]
    (Texto en la parte superior de la página. Ej: 'CONSERVADOR DE BIENES RAICES DE PUENTE ALTO', 'Copia Autorizada Fs 201v...')

    [Cuerpo central:]
    (Texto principal de la inscripción. Ej: 'Puente Alto, cinco de Diciembre...', 'TERESA DE LAS MERCEDES LOBOS MIRANDA...', 'Los adquirieron por sucesión...')

    [Nota al margen izquierdo:]
    (Texto que aparezca claramente en el margen izquierdo. Ej: 'Nº 317', 'HERENCIA', 'Repertorio 11374' etc.)

    [Nota al margen derecho:]
    (Texto que aparezca claramente en el margen derecho. Ej: 'Según Certificado del Servicio de Impuestos Internos...')

    [Sello circular izquierdo:]
    (Si aparece un sello en esta página, transcribe su texto aquí)
    
    [Sello circular derecho:]
    (Si aparece un sello en esta página, transcribe su texto aquí)

    [Nota inferior:]
    (Texto en la parte inferior de la página. Ej: 'CERTIFICO: Que la presente copia es testimonio fiel de su original...')

    [Firma electrónica:]
    (Texto relacionado con la firma digital o validación. Ej: 'Certificado emitido con Firma Electronica Avanzada...')

    [manuscrita:]
    (Cualquier texto escrito a mano. Ej: [manuscrita: firma ilegible])

    INSTRUCCIONES IMPORTANTES:
    **- IMPORTANTE: Tu respuesta debe comenzar *directamente* con la primera etiqueta (ej: `[Encabezado:]`). NO incluyas ninguna introducción, saludo o frase previa como 'Claro, aquí tienes...'.**
    - Si un bloque NO aparece en la imagen, escribe justo debajo del bloque: (sin contenido detectado)
    - "Encabezado" es la línea o líneas de arriba del todo (“CONSERVADOR DE BIENES RAÍCES...”, “Copia autorizada...”, etc.).
    - "Cuerpo central" es el texto jurídico principal de la inscripción (los nombres, cédulas, descripción del derecho de aguas, fojas, etc.).
    - "Nota inferior" es el párrafo final que suele decir “CERTIFICO: Que la presente copia...”.
    - "Firma electrónica" es el bloque de la firma digital (nombre del conservador, fecha, razón, ubicación). Si no viene separado, déjalo en Nota inferior.
    - Mantén tildes, puntos, guiones y mayúsculas tal como aparecen.
    - No agregues comentarios tuyos.
    - No consideres marcas de agua.
    - No elimines marcas como “Puente Alto, cinco de Diciembre de dos mil veinticuatro.-”.
    - El documento está en Chile.
    """
    

def extract_text_from_scanned_pdf_labeled(pdf_path: str) -> str:
    """
    Convierte cada página de un PDF en imagen y usa Gemini para extraer el texto,
    pero YA etiquetado según la estructura del CBR.
    """
    if not os.path.exists(pdf_path):
        return f"Error: El archivo no se encuentra en la ruta: {pdf_path}"

    print(f"Procesando el archivo: {pdf_path}...")

    # Se mantiene el modelo que te funciona
    model = genai.GenerativeModel('gemini-2.5-pro')

    doc = fitz.open(pdf_path)
    full_text_by_pages = []

    total_pages = len(doc)

    for page_num, page in enumerate(doc):
        print(f"  - Procesando página {page_num + 1}/{total_pages}...")

        # render a imagen
        pix = page.get_pixmap(dpi=300)
        img_bytes = pix.tobytes("png")
        image = Image.open(io.BytesIO(img_bytes))

        # prompt específico por página (AHORA USA LOS PROMPTS AJUSTADOS)
        prompt = build_prompt_for_page(page_num, total_pages)

        try:
            response = model.generate_content([prompt, image])

            if response.parts:
                page_text = response.text
            else:
                page_text = f"[No se pudo extraer texto de la página {page_num + 1}]"
        except Exception as e:
            print(f"  - Ocurrió un error en la página {page_num + 1}: {e}")
            page_text = f"[Error al procesar la página {page_num + 1}: {e}]"

        full_text_by_pages.append(page_text)

    doc.close()
    print("¡Procesamiento completado!")

    # unimos todas las páginas con el separador que ya usabas
    return "\n\n--- Nueva Página ---\n\n".join(full_text_by_pages)



print('')
print('✅ Función de extracción--OK.')
print('')


✅ Función de extracción--OK.

CPU times: user 326 μs, sys: 168 μs, total: 494 μs
Wall time: 475 μs


## Extracción

In [9]:
registros = []

# pdf_files = glob(os.path.join(data_path, "*.pdf"))
pdf_files = glob.glob(os.path.join(data_path, "*.[Pp][Dd][Ff]"))

print(f"Se encontraron {len(pdf_files)} archivos PDF")

for pdf_file in pdf_files:
    pdf_name = os.path.splitext(os.path.basename(pdf_file))[0]
    print(f"Procesando: {pdf_name}")

    try:
        extracted_content = extract_text_from_scanned_pdf_labeled(pdf_file)

        # guardamos TXT con el mismo nombre
        with open(os.path.join(result_path, f"{pdf_name}.txt"), "w", encoding="utf-8") as f:
            f.write(extracted_content)

        registros.append({
            "archivo": pdf_name,
            "contenido": extracted_content
        })

        print(f"✓ Completado: {pdf_name}")
    except Exception as e:
        print(f"✗ Error procesando {pdf_name}: {str(e)}")

contenido = pd.DataFrame(registros)
contenido.to_excel(os.path.join(result_path, "resultados_extraccion_etiquetada.xlsx"), index=False)

print("")
print("✅ Extracción finalizada con etiquetas.")
print("")

Se encontraron 228 archivos PDF
Procesando: SANTIAGO 2024 387 310
Procesando el archivo: /Volumes/D128_01/DGA_DARH_IA_CBR/CBR/Metropolitana/SANTIAGO/SANTIAGO 2024 387 310.pdf...
  - Procesando página 1/2...
  - Procesando página 2/2...
¡Procesamiento completado!
✓ Completado: SANTIAGO 2024 387 310
Procesando: SANTIAGO 2024 403 318
Procesando el archivo: /Volumes/D128_01/DGA_DARH_IA_CBR/CBR/Metropolitana/SANTIAGO/SANTIAGO 2024 403 318.pdf...
  - Procesando página 1/2...
  - Procesando página 2/2...
¡Procesamiento completado!
✓ Completado: SANTIAGO 2024 403 318
Procesando: SANTIAGO 2024 397 315
Procesando el archivo: /Volumes/D128_01/DGA_DARH_IA_CBR/CBR/Metropolitana/SANTIAGO/SANTIAGO 2024 397 315.pdf...
  - Procesando página 1/2...
  - Procesando página 2/2...
¡Procesamiento completado!
✓ Completado: SANTIAGO 2024 397 315
Procesando: SANTIAGO 2024 383 308
Procesando el archivo: /Volumes/D128_01/DGA_DARH_IA_CBR/CBR/Metropolitana/SANTIAGO/SANTIAGO 2024 383 308.pdf...
  - Procesando página